# Lab 12: The Invisible Overshoot

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 2, *The Time Editor*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-12.ipynb)

**What you will do:** measure with a ruler how far ahead of a ball's true stopping point you point when it disappears, turn that overshoot into a speed relationship, and watch a Kalman filter make (and correct) the same kind of forward prediction.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 12 you watched a ball roll across a table and, the instant your friend covered it,
pointed to where it had been. Your pointing consistently landed slightly *ahead* of the
ball's true last position, further along its path, and the faster the ball moved, the
larger that forward displacement became. This is **representational momentum**
(Freyd & Finke, 1984): the brain extrapolates a moving object's position forward to
compensate for the roughly 50-80 ms it takes visual signals to reach conscious awareness, so
when the object suddenly stops, the prediction overshoots.

## Record your results

For each trial, measure with the ruler how far ahead of the ball's actual stopping point
you pointed (in centimetres), and note roughly how fast the ball was rolling
(slow / medium / fast). Run several trials at each speed.

In [ ]:
# example data: replace with your own ruler measurements
trials = pd.DataFrame({
    "speed": ["slow", "slow", "slow", "medium", "medium", "medium", "fast", "fast", "fast"],
    "overshoot_cm": [1.2, 0.8, 1.5, 2.6, 3.1, 2.4, 4.8, 5.5, 4.2],
})
display(trials)
summary = trials.groupby("speed")["overshoot_cm"].agg(["mean", "std"]).reindex(["slow", "medium", "fast"])
display(summary)
summary["mean"].plot.bar(yerr=summary["std"], color="steelblue", capsize=4, figsize=(5, 3))
plt.ylabel("mean forward overshoot (cm)"); plt.xticks(rotation=0); plt.show()

## Compare

Lab 12 does not give a published number of centimetres or milliseconds to match here; the
effect is described only by its direction and its relationship with speed: "the faster the
ball moves, the larger the forward displacement." So the fair comparison is not with an
external value but with that predicted relationship, using your own data across the three
speeds.

In [ ]:
order = ["slow", "medium", "fast"]
means = summary["mean"].reindex(order)
increasing = all(means.iloc[i] <= means.iloc[i + 1] for i in range(len(means) - 1))
print("Mean overshoot by speed:", dict(means.round(1)))
if increasing:
    print("Your data follow the pattern the lab describes: overshoot grows with speed.")
else:
    print("Your data do not show a clean increase with speed. With only a few trials per speed,"
          " this is common: representational momentum is a small, noisy effect at the single-trial"
          " level. Averaging more trials, or using the slow-motion video from Step 3, would help.")

## The AI side

The **Kalman filter** (Kalman, 1960), already named in Lab 12's AI Connection, is the
engineering version of what your brain just did: it tracks a moving object from noisy,
delayed measurements by blending a prediction (based on the object's last known velocity)
with each new measurement, weighted by how much it trusts each one. Below, a dot moves at
constant velocity but its measured position is corrupted by noise. Watch how the filter's
*predicted* position runs ahead of the noisy measurement, and how far it overshoots when
the dot suddenly stops, the same overshoot you measured with the ball.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n_steps = 40
stop_at = 25          # the dot stops moving here, like the ball being covered
true_pos = np.concatenate([np.arange(0, stop_at) * 2.0, np.full(n_steps - stop_at, (stop_at - 1) * 2.0)])
measured = true_pos + rng.normal(0, 1.5, n_steps)   # noisy sensor

# A minimal 1-D Kalman filter tracking position and velocity
x = np.array([0.0, 2.0])              # [position, velocity]
P = np.eye(2) * 10
F = np.array([[1, 1], [0, 1]])        # constant-velocity motion model
H = np.array([[1, 0]])                # we only measure position
Q = np.eye(2) * 0.01                  # process noise (trust in the motion model)
R = np.array([[2.0]])                 # measurement noise (trust in the sensor)

predicted, corrected = [], []
for z in measured:
    x = F @ x                                  # predict
    P = F @ P @ F.T + Q
    predicted.append(x[0])
    y = z - H @ x                              # correct with the new measurement
    S = H @ P @ H.T + R
    K = P @ H.T @ np.linalg.inv(S)
    x = x + (K @ y)
    P = (np.eye(2) - K @ H) @ P
    corrected.append(x[0])

df = pd.DataFrame({"true": true_pos, "measured": measured, "predicted": predicted, "corrected": corrected})
display(df.iloc[stop_at - 3: stop_at + 5].round(2))

plt.figure(figsize=(7, 3.5))
plt.plot(df["true"], "k--", label="true position")
plt.plot(df["measured"], ".", color="grey", alpha=0.6, label="noisy measurement")
plt.plot(df["predicted"], color="firebrick", label="filter's prediction (before correction)")
plt.axvline(stop_at, color="steelblue", linestyle=":", label="object stops")
plt.xlabel("time step"); plt.ylabel("position"); plt.legend(fontsize=8); plt.show()

Look at the "predicted" line just after the object stops: it keeps climbing for a step or
two, overshooting the true position, before the correction pulls it back, the same brief,
forward-biased error you measured yourself. The disanalogy matters too: the Kalman filter
overshoots because its explicit motion model assumes constant velocity, and it corrects the
moment a new measurement arrives; your brain's overshoot is a one-shot perceptual estimate
with no later "measurement" to correct it, which is exactly why the illusion persists even
once you know the ball has stopped.

## Pool the class data

Pool the class's overshoot measurements across speeds to see whether the speed
relationship holds up with more data than one person's trials can show. Each row is one
person's mean overshoot (cm) at one speed, from an anonymous ID.

In [ ]:
import io
csv_text = """id,speed,overshoot_cm
P01,slow,1.1
P01,medium,2.8
P01,fast,4.9
P02,slow,0.6
P02,medium,2.1
P02,fast,3.7
P03,slow,1.8
P03,medium,3.4
P03,fast,5.6
P04,slow,0.9
P04,medium,2.5
P04,fast,4.3
"""  # example data: replace with your class CSV, e.g. pd.read_csv("overshoot_class.csv")
class_df = pd.read_csv(io.StringIO(csv_text))
order = ["slow", "medium", "fast"]

boots = {}
for sp in order:
    vals = class_df.loc[class_df.speed == sp, "overshoot_cm"]
    boot = [vals.sample(len(vals), replace=True).mean() for _ in range(5000)]
    boots[sp] = np.percentile(boot, [2.5, 50, 97.5])
boot_df = pd.DataFrame(boots, index=["2.5%", "median", "97.5%"]).T.reindex(order)
display(boot_df.round(2))

class_df.boxplot(column="overshoot_cm", by="speed", positions=[0, 1, 2], grid=False)
plt.suptitle(""); plt.title("Class overshoot by speed"); plt.ylabel("overshoot (cm)"); plt.show()
print(f"n = {class_df['id'].nunique()} people, {len(class_df)} speed x person cells")

## Questions to think about

1. Your Kalman filter overshoots for a step or two after the object stops, then corrects. Your own pointing overshoot is never corrected within the trial. What is different about the information available to the filter (a new measurement each step) versus the information available to your visual system (the ball is now hidden)?
2. Berry et al. (1999), cited in the lab, found that anticipation of moving stimuli begins in the retina itself, before the signal even reaches the brain. Does that change how you think about where the "prediction" in representational momentum happens?
3. If you increase the measurement noise (R) in the Kalman filter code, its position estimate should lag more, similar to how Lab 14 describes weak signals producing slower speed perception. Try it -- does a noisier "measurement" make the filter's predicted line track the true position more sluggishly?
4. The lab compares representational momentum to a self-driving car's Kalman filter failing when a pedestrian "suddenly turns" rather than stopping. Based on the code above, what do you predict would happen to the filter's error if, instead of stopping, the object reversed direction?

## Challenge

Use the phone slow-motion method from the Smartphone version: film your friend covering the
ball with their hand at an unpredictable moment while you point, then pause the slow-motion
replay at the exact frame the hand first covers the ball and measure, from the video, where
the ball actually was. Compare this video-based measurement with your live-pointing
measurements above: do they agree?

In [ ]:
# Example data: replace with measurements from your slow-motion video, in centimetres.
live_pointing_cm = [1.2, 0.8, 1.5, 2.6, 3.1, 2.4]     # example data: your Step 2 trials
video_measured_cm = [1.0, 0.9, 1.3, 2.3, 2.9, 2.6]    # example data: same trials, from slow-motion video

diffs = np.array(live_pointing_cm) - np.array(video_measured_cm)
print(f"Mean difference (live pointing minus video-measured): {diffs.mean():.2f} cm")
print("A small difference suggests your live pointing and the slow-motion video are measuring the same")
print("thing; a large or systematic difference may mean your pointing was also slowed by motor response")
print("time, not just by the perceptual overshoot the lab describes.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-12.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")